# 2. The Container Stacking Rules Problem

## Tier 2 — Priority-based constructive heuristic (runnable)

Tier 1 defined what “optimal” means (minimizing reshuffles) but relied on an exact approach that only works for very small instances.

Tier 2 implements a **constructive heuristic**: we process containers (arrival order) and decide where to place each one using a **score**.

### Learning goals

- Build a simple but practical stacking rule from scratch.
- Understand how scoring criteria (blocking risk, height balance, weight stability) change decisions.
- Produce a step-by-step placement log that an operator / student can follow.

### What this notebook outputs

- A step-by-step placement table (container → chosen stack/tier → score details)
- Final stack configuration
- Total reshuffles (blocking pairs)
- Logistics-focused visuals: stack diagram + blocking arrows + reshuffle breakdown

### Notes about the source example

The source narrative gives explicit **departure times** for the small 5-container example but does not specify weights.

- We keep departure times exactly as described.
- We add small illustrative weights (documented in the code) so that the “weight stability” term can be demonstrated.

In [1]:
# Environment check (no installs here)
#
# Best practice: dependencies should be preinstalled in the JupyterHub/Docker image.
# If you're running locally, install packages once in your Python environment.

from typing import List, Dict, Tuple, Optional

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")

## Concrete example (5 containers, 3 stacks, 2 tiers)

We use the same departure times from the narrative:

- C1(d=3), C2(d=1), C3(d=5), C4(d=2), C5(d=6)

We also assign simple example weights (tons) **only to demonstrate the weight-stability criterion**:

- C1(w=22), C2(w=20), C3(w=18), C4(w=25), C5(w=16)

You can change these weights and observe how the heuristic decisions change.

In [2]:
# ----------------------------
# Example data
# ----------------------------
# Each container has:
# - id
# - departure time d (smaller means it must leave earlier)
# - weight w (tons)

containers = [
    {"id": "C1", "d": 3, "w": 22},
    {"id": "C2", "d": 1, "w": 20},
    {"id": "C3", "d": 5, "w": 18},
    {"id": "C4", "d": 2, "w": 25},
    {"id": "C5", "d": 6, "w": 16},
]

NUM_STACKS = 3
MAX_TIERS = 2

# Lookup maps for convenience
cid_to_d = {c["id"]: c["d"] for c in containers}
cid_to_w = {c["id"]: c["w"] for c in containers}

containers

In [ ]:
# ----------------------------
# Core helper: reshuffle counting (same logic as Tier 1)
# ----------------------------
# We keep the same simple metric:
# Each blocking pair (lower departs earlier than an upper container in the same stack)
# is counted as 1 expected reshuffle.


def count_blocking_pairs_in_stack(stack_bottom_to_top: List[str]) -> int:
    reshuffles = 0
    for lower_idx in range(len(stack_bottom_to_top)):
        for upper_idx in range(lower_idx + 1, len(stack_bottom_to_top)):
            lower = stack_bottom_to_top[lower_idx]
            upper = stack_bottom_to_top[upper_idx]
            if cid_to_d[lower] < cid_to_d[upper]:
                reshuffles += 1
    return reshuffles


def count_total_reshuffles(stack_config: List[List[str]]) -> int:
    return sum(count_blocking_pairs_in_stack(stack) for stack in stack_config)


# Quick sanity check
count_total_reshuffles([["C2", "C1"], [], []])  # C1 blocks C2 => 1

## Heuristic scoring function

For each arriving container, we try each non-full stack and compute a **score**.

- Lower score is better.
- The stack with the smallest score is chosen.

We implement the criteria described in the practice-style specification:

- **Departure time violations (+10)**
  - Penalize placing a container that departs early on top of containers that depart later.
- **Stack height imbalance (+2)**
  - Prefer balanced utilization across stacks.
- **Weight instability (+5)**
  - Penalize placing a heavier container on top of a lighter one.

This is intentionally simple and transparent so you can follow every decision.

In [ ]:
# ----------------------------
# Scoring function (beginner-friendly)
# ----------------------------
# Input:
# - arriving container dict
# - candidate stack index
# - current stack configuration
# - penalty weights
#
# Output:
# - total score
# - and a breakdown dict (for logging/learning)


def placement_score(
    arriving: Dict,
    stack_idx: int,
    stack_config: List[List[str]],
    w_departure_violation: float = 10.0,
    w_height_imbalance: float = 2.0,
    w_weight_instability: float = 5.0,
) -> Tuple[float, Dict]:
    # Read arriving container properties
    cid = arriving["id"]
    d_new = arriving["d"]
    w_new = arriving["w"]

    # Candidate stack we are evaluating
    stack = stack_config[stack_idx]

    # ----------------------------
    # 1) Departure time violation penalty
    # ----------------------------
    # Count how many containers in this stack depart later than the arriving one.
    # If the arriving one departs early, putting it on top of later-departing ones
    # increases blocking risk.
    violation_count = 0
    for existing_cid in stack:
        if d_new < cid_to_d[existing_cid]:
            violation_count += 1

    violation_pen = w_departure_violation * float(violation_count)

    # ----------------------------
    # 2) Height imbalance penalty
    # ----------------------------
    # We prefer to keep stack heights similar.
    heights = [len(s) for s in stack_config]
    avg_h = float(np.mean(heights))

    # If we place into this stack, its height would become:
    new_h = len(stack) + 1

    height_pen = w_height_imbalance * abs(new_h - avg_h)

    # ----------------------------
    # 3) Weight instability penalty
    # ----------------------------
    # If the arriving container is heavier than the current top, penalize.
    weight_pen = 0.0
    if len(stack) > 0:
        top_cid = stack[-1]
        top_w = cid_to_w[top_cid]
        if w_new > top_w:
            weight_pen = w_weight_instability

    # Total score (lower is better)
    total = float(violation_pen + height_pen + weight_pen)

    breakdown = {
        "violation_count": int(violation_count),
        "violation_pen": float(violation_pen),
        "height_pen": float(height_pen),
        "weight_pen": float(weight_pen),
        "total": float(total),
    }

    return total, breakdown


# Quick check: score on empty vs non-empty stack
empty_config = [[], [], []]
non_empty_config = [["C2"], [], []]
placement_score({"id": "C1", "d": 3, "w": 22}, 0, empty_config), placement_score({"id": "C1", "d": 3, "w": 22}, 0, non_empty_config)

## Run the constructive heuristic (with a step-by-step log)

We process containers in arrival order.

At each step:

- Evaluate all non-full stacks.
- Pick the stack with the smallest score.
- Place the container on top.

We also record a breakdown of score components for transparency so a beginner can see **why** a stack was chosen.

In [ ]:
# ----------------------------
# Heuristic algorithm
# ----------------------------
# We implement the constructive heuristic described in Tier 2.
#
# Output:
# - final `stack_config`
# - a step-by-step `log_df`
# - total reshuffles (blocking pairs)


def priority_based_stacking(
    containers: List[Dict],
    num_stacks: int,
    max_tiers: int,
    w_departure_violation: float = 10.0,
    w_height_imbalance: float = 2.0,
    w_weight_instability: float = 5.0,
):
    # Current stack configuration: list of stacks, each stack is bottom -> top.
    stack_config: List[List[str]] = [[] for _ in range(num_stacks)]

    # Placement log: one row per container placement
    log_rows = []

    # Process in arrival order
    for step_idx, arriving in enumerate(containers, start=1):
        cid = arriving["id"]

        best_stack = None
        best_score = float("inf")
        best_breakdown = None

        # Try placing into each stack
        for s_idx in range(num_stacks):
            # Skip full stacks
            if len(stack_config[s_idx]) >= max_tiers:
                continue

            score, breakdown = placement_score(
                arriving,
                s_idx,
                stack_config,
                w_departure_violation=w_departure_violation,
                w_height_imbalance=w_height_imbalance,
                w_weight_instability=w_weight_instability,
            )

            # Keep the lowest score
            if score < best_score:
                best_score = score
                best_stack = s_idx
                best_breakdown = breakdown

        assert best_stack is not None
        assert best_breakdown is not None

        # Place container on top of the chosen stack
        tier = len(stack_config[best_stack])  # tier 0 = bottom
        stack_config[best_stack].append(cid)

        # Log decision
        log_rows.append(
            {
                "step": step_idx,
                "container": cid,
                "d": arriving["d"],
                "w": arriving["w"],
                "chosen_stack": best_stack + 1,  # stacks shown as 1..S
                "tier": tier,
                "violation_count": best_breakdown["violation_count"],
                "violation_pen": best_breakdown["violation_pen"],
                "height_pen": best_breakdown["height_pen"],
                "weight_pen": best_breakdown["weight_pen"],
                "score": best_breakdown["total"],
            }
        )

    total_reshuffles = count_total_reshuffles(stack_config)

    return stack_config, pd.DataFrame(log_rows), int(total_reshuffles)


# Run heuristic
stack_config, log_df, total_reshuffles = priority_based_stacking(containers, NUM_STACKS, MAX_TIERS)

print("Final stack configuration:", stack_config)
print("Total reshuffles (blocking pairs):", total_reshuffles)

log_df

## Visualization (logistics-focused)

We visualize the heuristic output in three ways:

- Stack diagram
- Blocking arrows (who blocks whom)
- Reshuffles by container (retrieval order) + cumulative reshuffles

These visuals help validate the stacking rule and communicate it to non-coders.

In [ ]:
# ----------------------------
# Build blocking table for visualization
# ----------------------------
# A row means: "upper blocks lower" within the same stack.


def blocking_pairs(stack_config: List[List[str]]) -> pd.DataFrame:
    pairs = []

    for s_idx, stack in enumerate(stack_config, start=1):
        # `stack` is bottom -> top
        for lower_idx in range(len(stack)):
            for upper_idx in range(lower_idx + 1, len(stack)):
                lower = stack[lower_idx]
                upper = stack[upper_idx]

                # If lower departs earlier, upper blocks lower.
                if cid_to_d[lower] < cid_to_d[upper]:
                    pairs.append(
                        {
                            "stack": s_idx,
                            "lower": lower,
                            "upper": upper,
                            "lower_departure": cid_to_d[lower],
                            "upper_departure": cid_to_d[upper],
                        }
                    )

    return pd.DataFrame(pairs)


blocking_df = blocking_pairs(stack_config)
blocking_df

In [ ]:
# ----------------------------
# Plot stack diagram + blocking arrows
# ----------------------------

fig, ax = plt.subplots(figsize=(8, 4))

stack_width = 1.0
cell_height = 1.0
x_gap = 0.6

stack_x = {s_idx: (s_idx - 1) * (stack_width + x_gap) for s_idx in range(1, NUM_STACKS + 1)}

# Draw empty slots
for s_idx in range(1, NUM_STACKS + 1):
    x0 = stack_x[s_idx]
    for tier in range(MAX_TIERS):
        rect = plt.Rectangle((x0, tier * cell_height), stack_width, cell_height, facecolor="#F2F4F7", edgecolor="#344054")
        ax.add_patch(rect)

# Labels
for s_idx, stack in enumerate(stack_config, start=1):
    x0 = stack_x[s_idx]
    for tier, cid in enumerate(stack):
        ax.text(
            x0 + stack_width / 2,
            tier * cell_height + cell_height / 2,
            f"{cid}\n(d={cid_to_d[cid]}, w={cid_to_w[cid]})",
            ha="center",
            va="center",
            fontsize=9,
            color="#101828",
        )

# Blocking arrows
if len(blocking_df) > 0:
    for _, row in blocking_df.iterrows():
        s_idx = int(row["stack"])
        lower = row["lower"]
        upper = row["upper"]

        stack = stack_config[s_idx - 1]
        lower_tier = stack.index(lower)
        upper_tier = stack.index(upper)

        x = stack_x[s_idx] + stack_width / 2
        y_lower = lower_tier * cell_height + cell_height / 2
        y_upper = upper_tier * cell_height + cell_height / 2

        ax.annotate(
            "",
            xy=(x, y_lower),
            xytext=(x, y_upper),
            arrowprops=dict(arrowstyle="->", lw=2, color="#F97066"),
        )

ax.set_title("Tier 2 heuristic — final stacking plan (with blocking arrows)")
ax.set_xticks([stack_x[s] + stack_width / 2 for s in range(1, NUM_STACKS + 1)])
ax.set_xticklabels([f"Stack {s}" for s in range(1, NUM_STACKS + 1)])
ax.set_yticks([])
ax.set_xlim(-0.5, stack_x[NUM_STACKS] + stack_width + 0.5)
ax.set_ylim(-0.2, MAX_TIERS * cell_height + 0.8)
plt.show()

In [ ]:
# ----------------------------
# Reshuffles per container + cumulative reshuffles
# ----------------------------
# We count reshuffles by asking: for each container, how many later-departing
# containers are stacked above it?

block_counts = {c["id"]: 0 for c in containers}

if len(blocking_df) > 0:
    for _, r in blocking_df.iterrows():
        block_counts[r["lower"]] += 1

# Retrieval order (increasing departure time)
retrieve_order = sorted([c["id"] for c in containers], key=lambda cid: cid_to_d[cid])
resh_per = [block_counts[cid] for cid in retrieve_order]

# Bar chart
plt.figure(figsize=(7, 3.2))
plt.bar(retrieve_order, resh_per, color="#2E90FA", edgecolor="#101828", alpha=0.85)
plt.title("Reshuffles needed per container (retrieval order)")
plt.xlabel("Container")
plt.ylabel("Reshuffles")
plt.grid(True, axis="y", alpha=0.25)
plt.show()

# Cumulative
cum = np.cumsum(resh_per)
plt.figure(figsize=(7, 3.2))
plt.plot(retrieve_order, cum, marker="o", lw=2, color="#344054")
plt.title("Cumulative reshuffles")
plt.xlabel("Container (retrieval order)")
plt.ylabel("Cumulative reshuffles")
plt.grid(True, alpha=0.25)
plt.show()

pd.DataFrame({"container": retrieve_order, "d": [cid_to_d[c] for c in retrieve_order], "reshuffles": resh_per})

## Solution quality check: how far are we from the Tier 1 optimum?

Tier 2 is a greedy heuristic, so it is not guaranteed to be optimal.

To make that concrete, we compute the **optimal reshuffle count** for this same small instance using a tiny exhaustive search (Tier 1 style), then report the gap.

This gives you a quantitative way to judge:

- solution quality (reshuffles)
- trade-off vs speed/interpretability


In [ ]:
# Compute the Tier 1 optimal reshuffles for the SAME small instance, then compare.
#
# This is feasible here because we only have 5 containers and 3 stacks with 2 tiers.

import itertools

arrival_order = [c["id"] for c in containers]


def apply_choice_sequence(choice_seq: Tuple[int, ...]) -> Optional[List[List[str]]]:
    stack_config: List[List[str]] = [[] for _ in range(NUM_STACKS)]
    for cid, stack_idx in zip(arrival_order, choice_seq):
        if len(stack_config[stack_idx]) >= MAX_TIERS:
            return None
        stack_config[stack_idx].append(cid)
    return stack_config


all_choice_sequences = list(itertools.product(range(NUM_STACKS), repeat=len(arrival_order)))

best_reshuffles = None
best_config = None

for seq in all_choice_sequences:
    cfg = apply_choice_sequence(seq)
    if cfg is None:
        continue
    resh = count_total_reshuffles(cfg)
    if best_reshuffles is None or resh < best_reshuffles:
        best_reshuffles = int(resh)
        best_config = cfg

assert best_reshuffles is not None

print("Heuristic reshuffles:", int(total_reshuffles))
print("Tier-1-style optimal reshuffles:", int(best_reshuffles))
print("Optimal stack config:", best_config)
print("Gap (heuristic - optimal):", int(total_reshuffles) - int(best_reshuffles))

## Why this Tier exists vs earlier Tiers (comparison)

### Why Tier 2 exists (vs Tier 1)

Tier 1 teaches what “optimal” stacking means, but exact optimization becomes expensive as the yard becomes more loaded.

Tier 2 exists to provide a rule that is:

- fast to compute
- easy to explain and operationalize
- good enough for real-time use

### Advantages vs Tier 1

- Much faster for larger instances (greedy scoring instead of enumerating many possibilities)
- Transparent and adjustable (penalty weights can be tuned)
- Produces a step-by-step decision log

### Disadvantages vs Tier 1

- Not guaranteed optimal
- Sensitive to penalty weights and chosen scoring criteria
- Can make short-sighted choices (local decisions that hurt global reshuffles)

### When to use Tier 2

- Real-time stacking decisions in a busy yard
- When you need an interpretable baseline before trying metaheuristics or RL
- When an exact solver is unavailable


## Common pitfalls

- If reshuffle counts look surprising:
  - Print `blocking_df` and compare to the stack diagram.
  - Remember: this Tier uses the simple “blocking pairs” metric.
- If decisions look unstable:
  - The penalty weights strongly influence choices. Try changing:
    - departure violation weight
    - height imbalance weight
    - weight instability weight

## Next steps

- Try larger instances (more containers, higher tiers) where exhaustive search is impossible.
- Compare this heuristic to Tier 3 (GA) and Tier 4 (RL) on the same random instances.
